# LoRA / PEFT Fine-Tuning on a Small Classifier

**Goal:** fine-tune DistilBERT on SST-2 with LoRA adapters, measure trainable parameters, evaluate validation accuracy/F1, and run a small rank ablation.

This is designed for Google Colab. Runtime: **GPU recommended**, but the default subset is small enough to debug on CPU.

Interview angle: you can explain how freezing most of the base model changes memory, trainable parameters, and experiment velocity.


In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.45" "datasets>=2.20" "evaluate>=0.4" "accelerate>=0.33" "peft>=0.12" scikit-learn pandas


In [ ]:
import inspect
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"

MAX_TRAIN_EXAMPLES = 1000
MAX_VALID_EXAMPLES = 300
MAX_LENGTH = 128

BASE_LR = 2e-4
EPOCHS = 1
BATCH_SIZE = 8
GRAD_ACCUM = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)

train_ds = raw["train"].shuffle(seed=SEED).select(range(MAX_TRAIN_EXAMPLES))
valid_ds = raw["validation"].shuffle(seed=SEED).select(range(MAX_VALID_EXAMPLES))

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize, batched=True)
valid_tok = valid_ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tok[0])


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

def trainable_parameter_table(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        "trainable_params": trainable,
        "total_params": total,
        "trainable_pct": 100 * trainable / total,
    }

def make_training_args(output_dir, learning_rate=BASE_LR, epochs=EPOCHS):
    kwargs = dict(
        output_dir=output_dir,
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=epochs,
        weight_decay=0.01,
        logging_steps=25,
        save_strategy="no",
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )
    # Transformers renamed evaluation_strategy to eval_strategy in newer versions.
    strategy_name = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
    kwargs[strategy_name] = "epoch"
    return TrainingArguments(**kwargs)


def trainer_tokenizer_kwargs(tokenizer):
    # Transformers 5 uses processing_class; Transformers 4 used tokenizer.
    name = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
    return {name: tokenizer}


In [ ]:
def build_lora_model(rank=8, alpha=16, dropout=0.05):
    base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=["q_lin", "v_lin"],
        modules_to_save=["pre_classifier", "classifier"],
    )
    model = get_peft_model(base, lora_config)
    model.print_trainable_parameters()
    return model

lora_model = build_lora_model(rank=8)
trainable_parameter_table(lora_model)


In [ ]:
trainer = Trainer(
    model=lora_model,
    args=make_training_args("lora-sst2-r8"),
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    **trainer_tokenizer_kwargs(tokenizer),
    data_collator=collator,
    compute_metrics=compute_metrics,
)

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start
metrics = trainer.evaluate()

result = {
    "method": "LoRA",
    "rank": 8,
    "learning_rate": BASE_LR,
    "train_examples": MAX_TRAIN_EXAMPLES,
    "valid_examples": MAX_VALID_EXAMPLES,
    "seconds": round(elapsed, 1),
    **trainable_parameter_table(lora_model),
    **{k.replace("eval_", ""): v for k, v in metrics.items() if k.startswith("eval_")},
}
pd.DataFrame([result])


In [ ]:
# Optional comparison: full fine-tuning is slower and uses more trainable parameters.
# Set this to True if your Colab runtime has enough time/VRAM.
RUN_FULL_FINETUNE = False

if RUN_FULL_FINETUNE:
    full_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    full_trainer = Trainer(
        model=full_model,
        args=make_training_args("full-sst2", learning_rate=2e-5),
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        **trainer_tokenizer_kwargs(tokenizer),
        data_collator=collator,
        compute_metrics=compute_metrics,
    )
    start = time.time()
    full_trainer.train()
    elapsed = time.time() - start
    full_metrics = full_trainer.evaluate()
    full_result = {
        "method": "Full fine-tune",
        "rank": None,
        "learning_rate": 2e-5,
        "seconds": round(elapsed, 1),
        **trainable_parameter_table(full_model),
        **{k.replace("eval_", ""): v for k, v in full_metrics.items() if k.startswith("eval_")},
    }
    display(pd.DataFrame([result, full_result]))
else:
    print("Skipped full fine-tuning. Flip RUN_FULL_FINETUNE=True to compare.")


In [ ]:
# Small ablation: compare LoRA rank while keeping data and hyperparameters fixed.
# For a faster dry run, use RANKS = [4, 8]. Add 16 when you have time.
RUN_RANK_ABLATION = True
RANKS = [4, 8]

ablation_rows = []
if RUN_RANK_ABLATION:
    for rank in RANKS:
        print(f"\n=== LoRA rank {rank} ===")
        model = build_lora_model(rank=rank, alpha=2 * rank)
        trainer = Trainer(
            model=model,
            args=make_training_args(f"lora-sst2-r{rank}"),
            train_dataset=train_tok,
            eval_dataset=valid_tok,
            **trainer_tokenizer_kwargs(tokenizer),
            data_collator=collator,
            compute_metrics=compute_metrics,
        )
        start = time.time()
        trainer.train()
        elapsed = time.time() - start
        metrics = trainer.evaluate()
        ablation_rows.append({
            "method": "LoRA",
            "rank": rank,
            "seconds": round(elapsed, 1),
            **trainable_parameter_table(model),
            **{k.replace("eval_", ""): v for k, v in metrics.items() if k.startswith("eval_")},
        })

ablation_df = pd.DataFrame(ablation_rows)
ablation_df


## Talk Track

- I froze the pretrained base model and trained small LoRA adapter matrices in attention projections.
- I measured the trainable-parameter fraction instead of only reporting accuracy.
- The ablation changes LoRA rank while holding data, model, and learning rate fixed, so the comparison is interpretable.
